# Practical 5B — Boosting and Voting Classifiers

Compare AdaBoost, Gradient Boosting, XGBoost and hard/soft voting using standard classification metrics.

**Run the cells from top to bottom.** The dataset is generated inside this notebook before the practical begins.

## 1. Install / import requirements

If a package is missing, run `!pip install pandas numpy matplotlib seaborn scikit-learn statsmodels openpyxl requests xgboost` in a notebook cell. Most environments already have the common packages.

**Note:** Practical 5B also needs `xgboost` (`!pip install xgboost`).

In [1]:
from pathlib import Path
Path('data').mkdir(exist_ok=True)


## 2. Create the required dataset

This is the relevant part of `make_data.py`, placed here so the notebook is self-contained.

In [2]:
# Create the Iris dataset used for classification.
from sklearn.datasets import load_iris
iris = load_iris(as_frame=True).frame.rename(columns={"target":"species"})
iris.to_csv("data/Iris.csv", index=False)
print("Created data/Iris.csv", iris.shape)


Created data/Iris.csv (150, 5)


## 3. Practical implementation

The following cells perform the analysis and show the output.

In [3]:
import pandas as pd
from sklearn.ensemble import AdaBoostClassifier, GradientBoostingClassifier, VotingClassifier
from xgboost import XGBClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

df = pd.read_csv("data/Iris.csv")
X, y = df.drop(columns="species"), df["species"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=1)

ada = AdaBoostClassifier(n_estimators=100, random_state=1)
gbm = GradientBoostingClassifier(random_state=1)
xgb = XGBClassifier(eval_metric="mlogloss", random_state=1)
models = {"AdaBoost": ada, "GBM": gbm, "XGBoost": xgb,
          "Hard voting": VotingClassifier([("a", ada), ("g", gbm), ("x", xgb)], voting="hard"),
          "Soft voting": VotingClassifier([("a", ada), ("g", gbm), ("x", xgb)], voting="soft")}

rows = []
for name, m in models.items():
    p = m.fit(X_train, y_train).predict(X_test)
    rows.append([name, accuracy_score(y_test, p), precision_score(y_test, p, average="macro"),
                 recall_score(y_test, p, average="macro"), f1_score(y_test, p, average="macro")])
print(pd.DataFrame(rows, columns=["Model", "Accuracy", "Precision", "Recall", "F1"]).round(3).to_string(index=False))

      Model  Accuracy  Precision  Recall    F1
   AdaBoost     0.956      0.956   0.956 0.956
        GBM     0.956      0.956   0.956 0.956
    XGBoost     0.933      0.933   0.937 0.934
Hard voting     0.956      0.956   0.956 0.956
Soft voting     0.956      0.956   0.956 0.956
